# VoleykoçAI: Ek Ödev: Kimlik eğitimi (Identity Fine-Tuning)

Bu çalışma birinci, ikinci ve üçüncü ödevden **bağımsız** ayrı bir süreç. Amaç modele voleybol bilgisi öğretmek değil; **kim olduğunu** öğretmek: adını, yaratıcısını, görevini ve sınırlarını.

Google Colab'da çalışır. `Runtime → Change runtime type → T4 GPU` seç.

| | |
|---|---|
| Temel model | `unsloth/Qwen3-4B-Instruct-2507` |
| Veri seti | `berkcangumusisik/voleykoc-identity-tr` (turkish + english) |
| Çıktı | `berkcangumusisik/voleykoc-identity-lora` |

Kimlik veri seti küçük ve tekrarlı olduğu için ana ödevden daha az adım eğitiyorum: fazlası modelin genel yeteneğini bozar.

## 1) Kurulum

In [ ]:
%pip install -q unsloth
%pip install -q --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "GPU yok. Runtime -> Change runtime type -> T4 GPU seçip yeniden başlat."
)
print(f"GPU: {torch.cuda.get_device_properties(0).name}")

## 2) Model ve kimlik testi (eğitim öncesi)

Eğitimden önce modele kim olduğunu soruyorum. Bu noktada Qwen olduğunu söylemesi bekleniyor: kimlik eğitiminin işe yaradığını bununla kanıtlayacağım.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 1024   # identity answers are short, long context is unnecessary
SEED = 1337
BASE_MODEL = "unsloth/Qwen3-4B-Instruct-2507"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

In [ ]:
KIMLIK_SORULARI = [
    "Senin adın ne?",
    "Seni kim geliştirdi?",
    "Görevin ne?",
    "Neler yapabilirsin?",
    "Who are you?",
    "Who created you?",
]

SYSTEM_TR = (
    "Sen VoleykoçAI'sın. Berkcan Gümüşışık tarafından geliştirilmiş, Türkçe "
    "konuşan bir voleybol antrenörlük asistanısın."
)


def sor(soru, max_new_tokens=160):
    mesajlar = [
        {"role": "system", "content": SYSTEM_TR},
        {"role": "user", "content": soru},
    ]
    girdi = tokenizer.apply_chat_template(
        mesajlar, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    cikti = model.generate(
        input_ids=girdi,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(cikti[0][girdi.shape[1]:], skip_special_tokens=True)


FastLanguageModel.for_inference(model)

once = {}
for soru in KIMLIK_SORULARI:
    once[soru] = sor(soru)
    print(f"\n### {soru}\n{once[soru]}")

## 3) LoRA ekle

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

## 4) Kimlik veri setini yükle

Veri setinde iki bölüm var: `turkish` ve `english`. İkisini birleştirip eğitiyorum ki model kimliğini iki dilde de tutarlı anlatsın.

In [ ]:
from datasets import concatenate_datasets, load_dataset

DATASET = "berkcangumusisik/voleykoc-identity-tr"

tr = load_dataset(DATASET, split="turkish")
en = load_dataset(DATASET, split="english")
ds = concatenate_datasets([tr, en]).shuffle(seed=SEED)
print(f"turkish={len(tr)}  english={len(en)}  toplam={len(ds)}")

In [ ]:
def bicimlendir(ornek):
    mesajlar = [{"role": "system", "content": ornek["system"]}]
    mesajlar += [
        {"role": m["role"], "content": m["content"]}
        for m in ornek["conversations"]
    ]
    return {
        "text": tokenizer.apply_chat_template(
            mesajlar, tokenize=False, add_generation_prompt=False
        )
    }


ds = ds.map(bicimlendir, remove_columns=ds.column_names)
print(ds[0]["text"])

## 5) Eğit (kısa)

`max_steps=60` ile sınırlı tutuyorum. Kimlik verisi küçük ve tekrarlı; uzun eğitim modelin voleybol bilgisini ve genel akıcılığını bozar (catastrophic forgetting).

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,
        output_dir="outputs-identity",
        report_to="none",
    ),
)

istatistik = trainer.train()
print(f"\nSüre: {istatistik.metrics['train_runtime'] / 60:.1f} dakika")
print(f"Son kayıp: {istatistik.metrics['train_loss']:.4f}")

## 6) Kimlik testi (eğitim sonrası)

Aynı sorular. Model artık kendini VoleykoçAI olarak tanıtmalı ve yaratıcısı olarak Berkcan Gümüşışık'ı söylemeli.

In [ ]:
FastLanguageModel.for_inference(model)

sonra = {}
for soru in KIMLIK_SORULARI:
    sonra[soru] = sor(soru)
    print(f"\n{'=' * 70}\n### {soru}\n")
    print(f"--- ÖNCE ---\n{once[soru]}\n")
    print(f"--- SONRA ---\n{sonra[soru]}")

In [ ]:
# A non-identity question: can the model still do its job?
# Identity training should not degrade general ability.
print(sor("5-1 rotasyon sistemi nasıl çalışır?", max_new_tokens=256))

In [ ]:
import json

with open("kimlik_karsilastirma.json", "w", encoding="utf-8") as fh:
    json.dump(
        [{"soru": s, "once": once[s], "sonra": sonra[s]} for s in KIMLIK_SORULARI],
        fh, ensure_ascii=False, indent=2,
    )
print("kimlik_karsilastirma.json yazıldı: indirip reports/ altına koy.")

## 7) Hugging Face'e yükle

Token'ı hücreye düz metin yazma. Soldaki 🔑 **Secrets** panelinden `HF_TOKEN` ekle ve "Notebook access" anahtarını aç.

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
REPO = "berkcangumusisik/voleykoc-identity-lora"

model.push_to_hub(REPO, token=HF_TOKEN)
tokenizer.push_to_hub(REPO, token=HF_TOKEN)

print(f"Yüklendi -> https://huggingface.co/{REPO}")